In [1]:
import os
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

In [2]:
from google.colab import drive
drive.mount('/content/data', force_remount=True)

ROOT_DIRECTORY = 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista'
IMAGES_DIRECTORY = ROOT_DIRECTORY + '/evaluation/processed'
SPLITS_CSV_PATH = ROOT_DIRECTORY + '/k-fold_split/typography_dataset_splits.csv'

Mounted at /content/data


In [3]:
def prepare_grouped_dataset_splits(dataset_dir, n_splits=5, test_size=0.15, random_state=42):
    """
    Reads the dataset and splits it ensuring that images from the same drawer (cod_XXX)
    stay together in the same split, avoiding data leakage.
    """
    dataset_path = Path(dataset_dir)
    data = []
    
    for class_dir in dataset_path.iterdir():
        if not class_dir.is_dir():
            continue
            
        label = class_dir.name
        
        for sub_dir in class_dir.iterdir():
            if not sub_dir.is_dir():
                continue
                
            font_code = sub_dir.name
            # Create a unique ID for each drawer (e.g., 'serif_cod_001')
            drawer_id = f"{label}_{font_code}"
            
            for img_path in sub_dir.glob('*.*'): 
                if img_path.is_file() and img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                    data.append({
                        'image_path': str(img_path),
                        'label': label,
                        'drawer_id': drawer_id
                    })
                    
    df = pd.DataFrame(data)
    
    if df.empty:
        raise ValueError("No images found. Please check the dataset directory path.")
        
    print(f"Total images: {len(df)}")
    print(f"Total unique drawers: {df['drawer_id'].nunique()}\n")
    
    # 1. Hold-out Test Set Split (Group-Aware)
    # GroupShuffleSplit ensures drawers are kept together
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(gss.split(df, groups=df['drawer_id']))
    
    train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    
    print(f"--- Global Split ---")
    print(f"Train/Val pool: {len(train_val_df)} images | {train_val_df['drawer_id'].nunique()} drawers")
    print(f"Hold-out Test set: {len(test_df)} images | {test_df['drawer_id'].nunique()} drawers\n")
    
    # 2. Stratified Group K-Fold for Train/Validation
    # Ensures drawers are kept together AND classes are balanced
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    folds = []
    print(f"--- K-Fold ({n_splits} splits) ---")
    
    for fold, (train_idx, val_idx) in enumerate(sgkf.split(train_val_df, train_val_df['label'], groups=train_val_df['drawer_id'])):
        train_fold = train_val_df.iloc[train_idx]
        val_fold = train_val_df.iloc[val_idx]
        
        folds.append({
            'train': train_fold,
            'val': val_fold
        })
        
        print(f"Fold {fold + 1} -> "
              f"Train: {len(train_fold)} imgs ({train_fold['drawer_id'].nunique()} drawers) | "
              f"Val: {len(val_fold)} imgs ({val_fold['drawer_id'].nunique()} drawers)")
        
    return folds, test_df


In [4]:
def save_splits_to_csv(cv_folds, test_df, output_filename="dataset_splits.csv"):
    """
    Consolidates the Test set and all K-Folds into a single CSV file.
    """
    all_records = []
    
    print("Formatting Test Set...")
    for _, row in test_df.iterrows():
        all_records.append({
            'image_path': row['image_path'],
            'label': row['label'],
            'drawer_id': row['drawer_id'],
            'fold': -1,         # -1 indicates it belongs to the global hold-out test set
            'split_type': 'test'
        })
        
    print("Formatting K-Folds...")
    for fold_idx, fold_data in enumerate(cv_folds):
        
        # Add Training data for this fold
        for _, row in fold_data['train'].iterrows():
            all_records.append({
                'image_path': row['image_path'],
                'label': row['label'],
                'drawer_id': row['drawer_id'],
                'fold': fold_idx,
                'split_type': 'train'
            })
            
        # Add Validation data for this fold
        for _, row in fold_data['val'].iterrows():
            all_records.append({
                'image_path': row['image_path'],
                'label': row['label'],
                'drawer_id': row['drawer_id'],
                'fold': fold_idx,
                'split_type': 'val'
            })
            
    # Convert to DataFrame and save
    final_df = pd.DataFrame(all_records)
    final_df.to_csv(output_filename, index=False)
    
    print(f"\nSuccess! Saved {len(final_df)} total records to '{output_filename}'")
    return final_df

In [5]:
if __name__ == "__main__":
    if os.path.exists(IMAGES_DIRECTORY):
        # 1. Generate splits
        cv_folds, test_dataset = prepare_grouped_dataset_splits(
            dataset_dir=IMAGES_DIRECTORY, 
            n_splits=5, 
            test_size=0.20 
        )
        
        # 2. Save them to disk
        save_splits_to_csv(cv_folds, test_dataset, output_filename=SPLITS_CSV_PATH)

Total images: 1464
Total unique drawers: 146

--- Global Split ---
Train/Val pool: 1131 images | 116 drawers
Hold-out Test set: 333 images | 30 drawers

--- K-Fold (5 splits) ---
Fold 1 -> Train: 919 imgs (92 drawers) | Val: 212 imgs (24 drawers)
Fold 2 -> Train: 908 imgs (93 drawers) | Val: 223 imgs (23 drawers)
Fold 3 -> Train: 937 imgs (94 drawers) | Val: 194 imgs (22 drawers)
Fold 4 -> Train: 868 imgs (92 drawers) | Val: 263 imgs (24 drawers)
Fold 5 -> Train: 892 imgs (93 drawers) | Val: 239 imgs (23 drawers)
Formatting Test Set...
Formatting K-Folds...

Success! Saved 5988 total records to 'typography_dataset_splits.csv'
